# Level 5: compare the maps

The entity is the map: which maps were played, which race they favor, how long the games last on
them, how fast the players expand on them and where the fights happen. The positions of the
events (`x`, `y` in `unit_born` and `unit_died`) are map coordinates, so a map is also a picture.

Every question is answered twice, with [polars](https://docs.pola.rs/) expressions and with
[DuckDB](https://duckdb.org/docs/) SQL over the same files; `s2.same()` checks that both return the
same rows, so each pair is a worked translation between the two.

Previous: [Level 4](./Level_4_Players.ipynb). Then: the [hero examples](../README.md#hero).

In [1]:
import polars as pl
import plotly.express as px

import snapshot as s2
import viz

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(60)

# The snapshot: `$S2_IPC_DIR`, or `ipcs` in the replay directory when written by swarmy.
IPC_DIR = s2.ipc_dir()
# polars reads the files lazily, one LazyFrame per file.
details = s2.scan(IPC_DIR, "details")
stats = s2.scan(IPC_DIR, "stats")
upgrades = s2.scan(IPC_DIR, "upgrades")
unit_born = s2.scan(IPC_DIR, "unit_born")
unit_died = s2.scan(IPC_DIR, "unit_died")
# DuckDB reads the same files as SQL views of the same names, plus `players` and `games_1v1`.
con = s2.duck(IPC_DIR)

# They read every unit_born file, so they are collected once and reused.
players = s2.players(IPC_DIR).collect().lazy()
games = s2.games(IPC_DIR).collect().lazy()
# The maps compared are the most played ones, with at least this many 1v1 games.
MIN_GAMES = s2.at_least(20)
MAX_MAPS = 20

## The maps of the snapshot

Games, when they were played and the median length, per map, for the most played maps.

In [2]:
maps = (
    games.group_by("title")
    .agg(
        pl.len().alias("games"),
        pl.col("ext_datetime").min().dt.date().alias("first_game"),
        pl.col("ext_datetime").max().dt.date().alias("last_game"),
        pl.col("minutes").median().alias("median_minutes"),
    )
    .filter(pl.col("games") >= MIN_GAMES)
    # The most played maps, a tie at the cut broken by the name, in polars and in SQL alike.
    .sort("games", "title", descending=[True, False])
    .head(MAX_MAPS)
    .collect()
)
MAPS = maps["title"].to_list()
maps

title,games,first_game,last_game,median_minutes
str,u32,date,date,f64
"""2000 Atmospheres LE""",1359,2021-04-07,2023-01-03,11.726935
"""Lightshade LE""",1163,2020-10-01,2021-10-23,10.97247
"""Romanticide LE""",1159,2020-09-27,2021-10-18,12.537946
"""Oxide LE""",1103,2020-09-30,2021-10-18,11.164435
"""Jagannatha LE""",1014,2020-09-27,2021-10-18,11.391369
"""Deathaura LE""",816,2020-07-29,2021-04-06,12.070312
"""Pillars of Gold LE""",717,2020-07-28,2021-04-06,11.670387
"""Blackburn LE""",626,2021-04-07,2022-07-20,10.985863
"""[ESL] Oceanborn""",523,2023-10-26,2024-11-05,11.157738


In [3]:
sql = con.sql(f"""
    SELECT title, count(*) AS games, min(ext_datetime)::DATE AS first_game, max(ext_datetime)::DATE AS last_game, median(minutes) AS median_minutes
    FROM games
    GROUP BY title
    HAVING count(*) >= {MIN_GAMES}
    ORDER BY games DESC, title
    LIMIT {MAX_MAPS}
""")
s2.same(maps, sql)

'same 20 rows, 5 columns'

## Which race a map favors

The win rate of the first race of each matchup on each map (`PvT` is the share the Protoss won),
as the distance from 50%: to the right the first race is favored, to the left the second. The
mirrors are left out and so are the map-matchups with few games. The win rate of 20 games has a
margin of about 11 points, so only the long bars mean something.

In [4]:
map_balance = (
    games.filter(pl.col("title").is_in(MAPS) & (pl.col("winner_race") != pl.col("loser_race")) & (pl.col("winner_race") != "?") & (pl.col("loser_race") != "?"))
    .group_by("title", "matchup")
    .agg(pl.len().alias("games"), (pl.col("winner_race") == pl.col("matchup").str.slice(0, 1)).mean().alias("first_race_win_rate"))
    .filter(pl.col("games") >= MIN_GAMES)
    .with_columns((pl.col("first_race_win_rate") - 0.5).alias("edge"))
    .sort("matchup", "edge")
    .collect()
)
map_balance.pivot(on="matchup", index="title", values="first_race_win_rate").sort("title")

title,PvT,PvZ,TvZ
str,f64,f64,f64
"""2000 Atmospheres LE""",0.474403,0.46732,0.464516
"""Beckett Industries LE""",0.631579,0.491071,0.491228
"""Berlingrad LE""",0.490196,0.57971,0.561644
"""Blackburn LE""",0.543478,0.441026,0.482759
"""Deathaura LE""",0.585366,0.530488,0.469697
"""Eternal Empire LE""",0.509091,0.522388,0.447368
"""Hardwire LE""",0.529412,0.552632,0.54023
"""Jagannatha LE""",0.544554,0.387597,0.462428
"""Ley Lines""",0.443182,0.465909,0.556701


In [5]:
sql = con.sql(f"""
    SELECT title, matchup, count(*) AS games, avg((winner_race = matchup[1])::INTEGER) AS first_race_win_rate,
           avg((winner_race = matchup[1])::INTEGER) - 0.5 AS edge
    FROM games
    WHERE title IN (SELECT unnest(?::VARCHAR[])) AND winner_race <> loser_race AND winner_race <> '?' AND loser_race <> '?'
    GROUP BY ALL
    HAVING count(*) >= {MIN_GAMES}
""", params=[MAPS])
s2.same(map_balance, sql)

'same 60 rows, 5 columns'

In [6]:
balance_colors = {"first race favored": viz.DIVERGING[0], "second race favored": viz.DIVERGING[-1]}
fig = px.bar(
    map_balance.with_columns(pl.when(pl.col("edge") >= 0).then(pl.lit("first race favored")).otherwise(pl.lit("second race favored")).alias("side")),
    x="edge", y="title", color="side", facet_col="matchup", orientation="h", hover_data=["games", "first_race_win_rate"],
    color_discrete_map=balance_colors, category_orders={"matchup": ["PvT", "PvZ", "TvZ"]},
    title="Win rate of the first race of the matchup per map, as the distance from 50%", labels={"edge": "win rate - 50%", "title": "map"},
    height=max(450, 24 * map_balance["title"].n_unique() + 160),
)
fig.update_xaxes(tickformat="+.0%")
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
viz.show(fig, "level_5_map_balance")

## How long the games last on each map

The distribution of the game length per map, as box plots sorted by the median: a map where the
games are short is a map where the early attacks work.

In [7]:
lengths = games.filter(pl.col("title").is_in(MAPS)).select("title", "minutes").collect()
order = maps.sort("median_minutes")["title"].to_list()
fig = px.box(lengths, x="minutes", y="title", orientation="h", category_orders={"title": order}, points=False,
             title="Game length per map, sorted by the median", labels={"title": "map", "minutes": "minutes"}, height=max(450, 24 * len(order) + 160))
fig.update_traces(marker_color=viz.SERIES[0], line_width=1.5)
viz.show(fig, "level_5_game_length")

In [8]:
quartiles = (
    lengths.lazy()
    .group_by("title")
    .agg(pl.col("minutes").quantile(0.25).alias("q25"), pl.col("minutes").median().alias("median"), pl.col("minutes").quantile(0.75).alias("q75"), pl.len().alias("games"))
    .sort("median")
    .collect()
)
sql = con.sql("""
    SELECT title, quantile_cont(minutes, 0.25) AS q25, median(minutes) AS median, quantile_cont(minutes, 0.75) AS q75, count(*) AS games
    FROM games
    WHERE title IN (SELECT unnest(?::VARCHAR[]))
    GROUP BY title
""", params=[MAPS])
# polars `quantile` interpolates with "nearest" by default, DuckDB's quantile_cont linearly; both the same with `interpolation="linear"`.
quartiles_linear = lengths.lazy().group_by("title").agg(pl.col("minutes").quantile(0.25, interpolation="linear").alias("q25"), pl.col("minutes").median().alias("median"), pl.col("minutes").quantile(0.75, interpolation="linear").alias("q75"), pl.len().alias("games")).collect()
print(s2.same(quartiles_linear, sql))
quartiles

same 20 rows, 5 columns


title,q25,median,q75,games
str,f64,f64,f64,u32
"""Ley Lines""",6.014881,9.298363,13.046131,379
"""Beckett Industries LE""",6.965774,10.50186,14.668899,386
"""[ESL] Inside and Out""",7.83631,10.565476,14.924851,325
"""Berlingrad LE""",7.973958,10.752232,14.864583,272
"""Lightshade LE""",8.125744,10.97247,14.349702,1163
"""Blackburn LE""",8.206101,10.985863,15.092262,626
"""[ESL] Gresvan""",8.948661,11.095982,16.043899,289
"""[ESL] Oceanborn""",8.785714,11.157738,14.68006,523
"""Oxide LE""",8.1875,11.164435,14.630208,1103


## How fast the players expand

The game time of the second and the third townhall of each player (the first and second
expansion), the median per map: the maps with a far natural or a hard third are slower. The
n-th value of a sorted group is `sort().get(n)` in polars and `QUALIFY row_number() = n` in SQL.

In [9]:
townhalls = (
    unit_born.filter((pl.col("ext_replay_loop") > 0) & pl.col("unit_type_name").is_in(s2.TOWNHALLS))
    .join(games.select("ext_fs_id", "title").filter(pl.col("title").is_in(MAPS)), on="ext_fs_id")
    .group_by("ext_fs_id", "title", "control_player_id")
    .agg(
        s2.game_seconds().sort().get(0, null_on_oob=True).alias("second_base"),
        s2.game_seconds().sort().get(1, null_on_oob=True).alias("third_base"),
    )
)
expansions = (
    townhalls.group_by("title")
    .agg(pl.len().alias("players"), pl.col("second_base").median().alias("median_second_base"), pl.col("third_base").median().alias("median_third_base"))
    .sort("median_third_base")
    .collect()
)
expansions

title,players,median_second_base,median_third_base
str,u32,f64,f64
"""Beckett Industries LE""",669,155.044643,299.776786
"""Hardwire LE""",556,156.473214,299.910714
"""[ESL] Alcyone""",827,166.607143,303.526786
"""Blackburn LE""",1152,157.03125,303.883929
"""[ESL] Site Delta""",681,162.991071,306.71875
"""[ESL] Data-C""",765,162.678571,310.714286
"""Berlingrad LE""",501,155.401786,312.455357
"""Jagannatha LE""",1900,158.303571,315.669643
"""2000 Atmospheres LE""",2579,161.785714,317.544643


In [10]:
sql = con.sql(f"""
    WITH ranked AS (
        SELECT b.ext_fs_id, g.title, b.control_player_id, game_seconds(b.ext_replay_loop) AS seconds,
               row_number() OVER (PARTITION BY b.ext_fs_id, b.control_player_id ORDER BY b.ext_replay_loop) AS n
        FROM unit_born b
        JOIN games g USING (ext_fs_id)
        WHERE b.ext_replay_loop > 0 AND b.unit_type_name IN ({s2.sql_list(s2.TOWNHALLS)}) AND g.title IN (SELECT unnest(?::VARCHAR[]))
    ),
    per_player AS (
        SELECT ext_fs_id, title, control_player_id,
               min(seconds) FILTER (WHERE n = 1) AS second_base, min(seconds) FILTER (WHERE n = 2) AS third_base
        FROM ranked
        GROUP BY ALL
    )
    SELECT title, count(*) AS players, median(second_base) AS median_second_base, median(third_base) AS median_third_base
    FROM per_player
    GROUP BY title
""", params=[MAPS])
s2.same(expansions, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 20 rows, 4 columns'

## Where the fights happen

The most played map as a picture: the units killed by a player, counted in cells of 8 by 8 map
units, over every game on the map, with the starting townhalls of all the games as squares (the
spawn positions). The darker the cell the more units died there. A binned count is a
`group_by` of the coordinates divided by the cell size, in both languages.

In [11]:
MAP = MAPS[0]
CELL = 8
map_games = games.filter(pl.col("title") == MAP).select("ext_fs_id")
EXPIRES = ["InvisibleTargetDummy", "Larva", "Egg", "Broodling", "BroodlingEscort", "AdeptPhaseShift", "ForceField", "KD8Charge", "CreepTumorBurrowed"]
cells = (
    unit_died.join(map_games, on="ext_fs_id", how="semi")
    .filter(pl.col("killer_player_id").is_not_null() & ~pl.col("unit_died_name").is_in(EXPIRES))
    .group_by(((pl.col("x").cast(pl.Int32) // CELL) * CELL).alias("cell_x"), ((pl.col("y").cast(pl.Int32) // CELL) * CELL).alias("cell_y"))
    .agg(pl.len().alias("deaths"))
    .sort("cell_x", "cell_y")
    .collect()
)
spawns = (
    unit_born.join(map_games, on="ext_fs_id", how="semi")
    .filter((pl.col("ext_replay_loop") == 0) & pl.col("unit_type_name").is_in(s2.TOWNHALLS))
    .group_by("x", "y")
    .agg(pl.len().alias("games"))
    .collect()
)
print(f"{MAP}: {cells['deaths'].sum()} units killed in {map_games.select(pl.len()).collect().item()} games, {spawns.height} spawn positions")
cells.sort("deaths", descending=True).head(5)

2000 Atmospheres LE: 504182 units killed in 1359 games, 2 spawn positions


cell_x,cell_y,deaths
i32,i32,u32
144,120,10005
144,152,9364
80,80,8777
112,56,8760
136,120,8369


In [12]:
sql = con.sql(f"""
    SELECT (x::INTEGER // {CELL}) * {CELL} AS cell_x, (y::INTEGER // {CELL}) * {CELL} AS cell_y, count(*) AS deaths
    FROM unit_died
    WHERE ext_fs_id IN (SELECT ext_fs_id FROM games WHERE title = ?) AND killer_player_id IS NOT NULL
      AND unit_died_name NOT IN ({s2.sql_list(EXPIRES)})
    GROUP BY ALL
""", params=[MAP])
s2.same(cells, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 306 rows, 3 columns'

In [13]:
import plotly.graph_objects as go

grid = cells.pivot(on="cell_x", index="cell_y", values="deaths", sort_columns=True).sort("cell_y").fill_null(0)
xs = [int(column) for column in grid.columns if column != "cell_y"]
fig = go.Figure(
    go.Heatmap(
        z=grid.drop("cell_y").to_numpy(), x=[x + CELL / 2 for x in xs], y=(grid["cell_y"] + CELL / 2).to_list(),
        colorscale=[[0, "#fcfcfb"], [0.001, viz.SEQUENTIAL[0]], [1, viz.SEQUENTIAL[-1]]], colorbar=dict(title="units died"),
        hovertemplate="x %{x}, y %{y}: %{z} units died<extra></extra>",
    )
)
fig.add_trace(go.Scatter(x=spawns["x"], y=spawns["y"], mode="markers", name="spawn", marker=dict(symbol="square", size=14, color=viz.SERIES[1], line=dict(width=2, color="#fcfcfb")),
                         hovertemplate="spawn at %{x}, %{y}<extra></extra>"))
fig.update_layout(title=f"{MAP}: where the units died, over every game on the map", width=720, height=680)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
viz.show(fig, "level_5_death_heatmap")

## What to take from Level 5

- The same measures as the players (win rate, length, expansions) grouped by `title`; the
  matchup is sorted so that `PvT` means one thing on every map.
- A win rate per map and matchup needs more games than one snapshot of a tournament usually has;
  the margin is in the text next to the chart, not hidden.
- The coordinates make the map a picture: bin, count, draw. The spawn positions are the
  townhalls born at loop 0.

Next: the [hero examples](../README.md#hero), full analyses built from these pieces.